In [16]:
from dask_setup import setup_dask_client 

from pathlib import Path
import glob
from zarr.codecs import BloscCodec

import xarray as xr

In [2]:
client, cluster, dask_tmp = setup_dask_client(mode="interactive", workload_type="cpu")   # heavy compute

INFO     [client] Interactive cluster mode — using already-allocated nodes
INFO     [multinode] Interactive cluster: single node, using LocalCluster (node=gadi-cpu-spr-0210.gadi.nci.org.au)
INFO     [client] Starting Dask client setup (workload_type=cpu | environment=jupyter)
INFO     [resources] Resources detected via PBS (total_cores=104 | total_mem_gib=496.0)


INFO     [client] Temp/spill dir: /jobfs/178014662.gadi-pbs/dask-1639476
INFO     [client] Workers: 104 | threads/worker: 1 | processes: True
INFO     [client] Mem: total ~496.0 GiB | usable ~446.0 GiB | per-worker ~4.3 GiB
INFO     [client] Compression: spill=auto | comm=False
INFO     [client] Dask client ready


[setup_dask_client] Configuration summary
temp/spill dir: /jobfs/178014662.gadi-pbs/dask-1639476
Workers: 104 | threads/worker: 1 | processes: True
Memory: total ~496.0 GiB | usable ~446.0 GiB | per-worker ~4.3 GiB
Compression: spill=auto | comm=False


2026-09-02 13:10:18,815 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle 5648d458985612f7695bdbb6b492fe1c initialized by task ('rechunk-merge-rechunk-transfer-58c73697dd63b170a3706ba75ae4a49a', 0, 0, 0, 5, 0, 0) executed on worker tcp://127.0.0.1:37475
2026-09-02 13:10:57,827 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle 5648d458985612f7695bdbb6b492fe1c deactivated due to stimulus 'task-finished-1788318655.1703045'
2026-09-02 13:16:15,783 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle 63edff8a29e1b3c009cc0b06de0b64d8 initialized by task ('rechunk-merge-rechunk-transfer-c1bf678d76538c3161ffa146ae751305', 0, 0, 0, 14, 0, 0) executed on worker tcp://127.0.0.1:37475
2026-09-02 13:16:55,748 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle 63edff8a29e1b3c009cc0b06de0b64d8 deactivated due to stimulus 'task-finished-1788319012.0164776'


In [3]:
%cd /g/data/w42/dr6273/work/wind_drought/
import functions as fn

%load_ext autoreload
%autoreload 2

/g/data/w42/dr6273/work/wind_drought


In [1]:
BARRA_R2_PATH = "/g/data/ob53/BARRA2/output/reanalysis/AUS-11/BOM/ERA5/historical/hres/BARRA-R2/v1/day/" # daily mean (?)

BARRA_R2_WRITE_PATH = "/g/data/ng72/dr6273/work/projects/wind_drought/data/BARRA-R2/"

### BARRA-R2 data

In [5]:
def barraR2_preprocess(ds):
    """ Return smaller region """
    ds = ds.sel(
        lon=slice(REGION[0], REGION[1]),
        lat=slice(REGION[3], REGION[2])
    )
    return ds

In [6]:
def open_barra(files):
    """
    Open multiple files and preprocess to region.
    """
    ds = xr.open_mfdataset(
        files,
        preprocess=barraR2_preprocess,
        chunks='auto',
        parallel=True
    )
    return ds

In [7]:
def get_files(path, var, start=197901, end=202512):
    """ Return files between start and end month """
    files = sorted(Path(path).glob(
        var + "/latest/*.nc"
    ))
    
    files = [
        f for f in files
        if start <= int(f.stem[-13:-7]) <= end
    ]
    
    return files

In [8]:
def doy_anom(ds):
    """ Day of year anomalies """
    return ds.groupby('time.dayofyear') - ds.groupby('time.dayofyear').mean()

In [57]:
def specify_encoding(ds):
    """ Return encoding dict """
    codec = BloscCodec(cname='zstd', clevel=6, shuffle='bitshuffle')

    chunks = tuple(ds.chunks[dim][0] for dim in ds.dims)

    encoding = {}
    for var in ds.data_vars:
        encoding[var] = {
            'chunks': chunks, #(366, 319, 5),
            'compressors': codec,
            'dtype': 'float64'
        }
    return encoding

In [10]:
def write_zarr(ds, path, filename, encoding):
    """ Write to zarr """
    ds.to_zarr(
        path + filename,
        mode='w',
        encoding=encoding,
        zarr_format=3,
        consolidated=False
    )

In [11]:
REGION = fn.get_Aus_boundary()
YEARS = range(1979, 2026)

2m air temperature

In [61]:
tas_files = get_files(BARRA_R2_PATH, 'tas')

In [63]:
tas = open_barra(tas_files).drop_vars('time_bnds')

In [64]:
tas

<xarray.Dataset> Size: 18GB
Dimensions:  (time: 17167, lat: 319, lon: 409)
Coordinates:
  * time     (time) datetime64[ns] 137kB 1979-01-01T12:00:00 ... 2025-12-31T1...
  * lat      (lat) float64 3kB -44.99 -44.88 -44.77 ... -10.23 -10.12 -10.01
  * lon      (lon) float64 3kB 110.0 110.2 110.3 110.4 ... 154.7 154.8 154.9
    height   float64 8B 1.5
    crs      int32 4B 0
Data variables:
    tas      (time, lat, lon) float64 18GB dask.array<chunksize=(31, 319, 409), meta=np.ndarray>
Attributes: (12/59)
    axiom_version:             0.1.0
    axiom_schemas_version:     0.1.0
    axiom_schema:              cordex-1D.json
    productive_version:        edfab29
    variable_version:          v20231001
    Conventions:               CF-1.10, ACDD-1.3
    ...                        ...
    geospatial_lat_max:        12.98
    geospatial_lat_units:      degrees_north
    geospatial_lon_min:        88.48
    geospatial_lon_max:        207.39
    geospatial_lon_units:      degrees_east
    history:                   Sat May 04 16:03:54 2024: /g/data/access/ngm/m...

In [65]:
tas_chunked = tas.chunk({"time": -1, "lat": -1, "lon": 5})

In [66]:
tas_chunked

<xarray.Dataset> Size: 18GB
Dimensions:  (time: 17167, lat: 319, lon: 409)
Coordinates:
  * time     (time) datetime64[ns] 137kB 1979-01-01T12:00:00 ... 2025-12-31T1...
  * lat      (lat) float64 3kB -44.99 -44.88 -44.77 ... -10.23 -10.12 -10.01
  * lon      (lon) float64 3kB 110.0 110.2 110.3 110.4 ... 154.7 154.8 154.9
    height   float64 8B 1.5
    crs      int32 4B 0
Data variables:
    tas      (time, lat, lon) float64 18GB dask.array<chunksize=(17167, 319, 5), meta=np.ndarray>
Attributes: (12/59)
    axiom_version:             0.1.0
    axiom_schemas_version:     0.1.0
    axiom_schema:              cordex-1D.json
    productive_version:        edfab29
    variable_version:          v20231001
    Conventions:               CF-1.10, ACDD-1.3
    ...                        ...
    geospatial_lat_max:        12.98
    geospatial_lat_units:      degrees_north
    geospatial_lon_min:        88.48
    geospatial_lon_max:        207.39
    geospatial_lon_units:      degrees_east
    history:                   Sat May 04 16:03:54 2024: /g/data/access/ngm/m...

In [67]:
tas_anoms = doy_anom(tas_chunked)

In [68]:
tas_anoms

<xarray.Dataset> Size: 18GB
Dimensions:    (time: 17167, lat: 319, lon: 409)
Coordinates:
  * time       (time) datetime64[ns] 137kB 1979-01-01T12:00:00 ... 2025-12-31...
    height     (time) float64 137kB 1.5 1.5 1.5 1.5 1.5 ... 1.5 1.5 1.5 1.5 1.5
    crs        (time) int32 69kB 0 0 0 0 0 0 0 0 0 0 0 ... 0 0 0 0 0 0 0 0 0 0 0
    dayofyear  (time) int64 137kB 1 2 3 4 5 6 7 ... 359 360 361 362 363 364 365
  * lat        (lat) float64 3kB -44.99 -44.88 -44.77 ... -10.23 -10.12 -10.01
  * lon        (lon) float64 3kB 110.0 110.2 110.3 110.4 ... 154.7 154.8 154.9
Data variables:
    tas        (time, lat, lon) float64 18GB dask.array<chunksize=(366, 319, 5), meta=np.ndarray>
Attributes: (12/59)
    axiom_version:             0.1.0
    axiom_schemas_version:     0.1.0
    axiom_schema:              cordex-1D.json
    productive_version:        edfab29
    variable_version:          v20231001
    Conventions:               CF-1.10, ACDD-1.3
    ...                        ...
    geospatial_lat_max:        12.98
    geospatial_lat_units:      degrees_north
    geospatial_lon_min:        88.48
    geospatial_lon_max:        207.39
    geospatial_lon_units:      degrees_east
    history:                   Sat May 04 16:03:54 2024: /g/data/access/ngm/m...

In [69]:
tas_anoms = tas_anoms.chunk({"time": -1, "lat": -1, "lon": 5})

In [70]:
tas_anoms

<xarray.Dataset> Size: 18GB
Dimensions:    (time: 17167, lat: 319, lon: 409)
Coordinates:
  * time       (time) datetime64[ns] 137kB 1979-01-01T12:00:00 ... 2025-12-31...
    height     (time) float64 137kB dask.array<chunksize=(17167,), meta=np.ndarray>
    crs        (time) int32 69kB dask.array<chunksize=(17167,), meta=np.ndarray>
    dayofyear  (time) int64 137kB dask.array<chunksize=(17167,), meta=np.ndarray>
  * lat        (lat) float64 3kB -44.99 -44.88 -44.77 ... -10.23 -10.12 -10.01
  * lon        (lon) float64 3kB 110.0 110.2 110.3 110.4 ... 154.7 154.8 154.9
Data variables:
    tas        (time, lat, lon) float64 18GB dask.array<chunksize=(17167, 319, 5), meta=np.ndarray>
Attributes: (12/59)
    axiom_version:             0.1.0
    axiom_schemas_version:     0.1.0
    axiom_schema:              cordex-1D.json
    productive_version:        edfab29
    variable_version:          v20231001
    Conventions:               CF-1.10, ACDD-1.3
    ...                        ...
    geospatial_lat_max:        12.98
    geospatial_lat_units:      degrees_north
    geospatial_lon_min:        88.48
    geospatial_lon_max:        207.39
    geospatial_lon_units:      degrees_east
    history:                   Sat May 04 16:03:54 2024: /g/data/access/ngm/m...

In [71]:
tas_encoding = specify_encoding(tas_anoms)

In [72]:
tas_encoding['tas']

{'chunks': (17167, 319, 5),
 'compressors': BloscCodec(_tunable_attrs={'typesize'}, typesize=1, cname=<BloscCname.zstd: 'zstd'>, clevel=6, shuffle=<BloscShuffle.bitshuffle: 'bitshuffle'>, blocksize=0),
 'dtype': 'float64'}

In [73]:
write_zarr(
    tas_anoms,
    BARRA_R2_WRITE_PATH,
    'tas_dayofyear_anoms_1979-2025.zarr',
    tas_encoding
)

2026-09-02 13:18:39,171 - distributed.worker - ERROR - Failed to communicate with scheduler during heartbeat.
Traceback (most recent call last):
  File "/g/data/xp65/public/apps/med_conda/envs/analysis3-26.08/lib/python3.12/site-packages/distributed/comm/tcp.py", line 226, in read
    frames_nosplit_nbytes_bin = await stream.read_bytes(fmt_size)
                                ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
tornado.iostream.StreamClosedError: Stream is closed

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/g/data/xp65/public/apps/med_conda/envs/analysis3-26.08/lib/python3.12/site-packages/distributed/worker.py", line 1273, in heartbeat
    response = await retry_operation(
               ^^^^^^^^^^^^^^^^^^^^^^
  File "/g/data/xp65/public/apps/med_conda/envs/analysis3-26.08/lib/python3.12/site-packages/distributed/utils_comm.py", line 416, in retry_operation
    return await retry(
           ^^^^^^^^^^^^
  File "/g/d

Surface downwelling shortwave radiation

In [14]:
rsds_files = get_files(BARRA_R2_PATH, 'rsds')

In [42]:
rsds = open_barra(rsds_files).drop_vars('time_bnds').drop_dims('bnds')

In [43]:
rsds

<xarray.Dataset> Size: 18GB
Dimensions:  (time: 17167, lat: 319, lon: 409)
Coordinates:
  * time     (time) datetime64[ns] 137kB 1979-01-01T12:00:00 ... 2025-12-31T1...
  * lat      (lat) float64 3kB -44.99 -44.88 -44.77 ... -10.23 -10.12 -10.01
  * lon      (lon) float64 3kB 110.0 110.2 110.3 110.4 ... 154.7 154.8 154.9
    crs      int32 4B 0
Data variables:
    rsds     (time, lat, lon) float64 18GB dask.array<chunksize=(31, 319, 409), meta=np.ndarray>
Attributes: (12/59)
    axiom_version:             0.1.0
    axiom_schemas_version:     0.1.0
    axiom_schema:              cordex-1D.json
    productive_version:        edfab29
    variable_version:          v20231001
    Conventions:               CF-1.10, ACDD-1.3
    ...                        ...
    geospatial_lat_max:        12.98
    geospatial_lat_units:      degrees_north
    geospatial_lon_min:        88.48
    geospatial_lon_max:        207.39
    geospatial_lon_units:      degrees_east
    history:                   Sat May 04 16:03:59 2024: /g/data/access/ngm/m...

In [44]:
rsds_chunked = rsds.chunk({"time": -1, "lat": -1, "lon": 5})

In [45]:
rsds_chunked

<xarray.Dataset> Size: 18GB
Dimensions:  (time: 17167, lat: 319, lon: 409)
Coordinates:
  * time     (time) datetime64[ns] 137kB 1979-01-01T12:00:00 ... 2025-12-31T1...
  * lat      (lat) float64 3kB -44.99 -44.88 -44.77 ... -10.23 -10.12 -10.01
  * lon      (lon) float64 3kB 110.0 110.2 110.3 110.4 ... 154.7 154.8 154.9
    crs      int32 4B 0
Data variables:
    rsds     (time, lat, lon) float64 18GB dask.array<chunksize=(17167, 319, 5), meta=np.ndarray>
Attributes: (12/59)
    axiom_version:             0.1.0
    axiom_schemas_version:     0.1.0
    axiom_schema:              cordex-1D.json
    productive_version:        edfab29
    variable_version:          v20231001
    Conventions:               CF-1.10, ACDD-1.3
    ...                        ...
    geospatial_lat_max:        12.98
    geospatial_lat_units:      degrees_north
    geospatial_lon_min:        88.48
    geospatial_lon_max:        207.39
    geospatial_lon_units:      degrees_east
    history:                   Sat May 04 16:03:59 2024: /g/data/access/ngm/m...

In [46]:
rsds_anoms = doy_anom(rsds_chunked)

In [47]:
rsds_anoms

<xarray.Dataset> Size: 18GB
Dimensions:    (time: 17167, lat: 319, lon: 409)
Coordinates:
  * time       (time) datetime64[ns] 137kB 1979-01-01T12:00:00 ... 2025-12-31...
    crs        (time) int32 69kB 0 0 0 0 0 0 0 0 0 0 0 ... 0 0 0 0 0 0 0 0 0 0 0
    dayofyear  (time) int64 137kB 1 2 3 4 5 6 7 ... 359 360 361 362 363 364 365
  * lat        (lat) float64 3kB -44.99 -44.88 -44.77 ... -10.23 -10.12 -10.01
  * lon        (lon) float64 3kB 110.0 110.2 110.3 110.4 ... 154.7 154.8 154.9
Data variables:
    rsds       (time, lat, lon) float64 18GB dask.array<chunksize=(366, 319, 5), meta=np.ndarray>
Attributes: (12/59)
    axiom_version:             0.1.0
    axiom_schemas_version:     0.1.0
    axiom_schema:              cordex-1D.json
    productive_version:        edfab29
    variable_version:          v20231001
    Conventions:               CF-1.10, ACDD-1.3
    ...                        ...
    geospatial_lat_max:        12.98
    geospatial_lat_units:      degrees_north
    geospatial_lon_min:        88.48
    geospatial_lon_max:        207.39
    geospatial_lon_units:      degrees_east
    history:                   Sat May 04 16:03:59 2024: /g/data/access/ngm/m...

In [48]:
rsds_anoms = rsds_anoms.chunk({"time": -1, "lat": -1, "lon": 5})

In [49]:
rsds_anoms

<xarray.Dataset> Size: 18GB
Dimensions:    (time: 17167, lat: 319, lon: 409)
Coordinates:
  * time       (time) datetime64[ns] 137kB 1979-01-01T12:00:00 ... 2025-12-31...
    crs        (time) int32 69kB dask.array<chunksize=(17167,), meta=np.ndarray>
    dayofyear  (time) int64 137kB dask.array<chunksize=(17167,), meta=np.ndarray>
  * lat        (lat) float64 3kB -44.99 -44.88 -44.77 ... -10.23 -10.12 -10.01
  * lon        (lon) float64 3kB 110.0 110.2 110.3 110.4 ... 154.7 154.8 154.9
Data variables:
    rsds       (time, lat, lon) float64 18GB dask.array<chunksize=(17167, 319, 5), meta=np.ndarray>
Attributes: (12/59)
    axiom_version:             0.1.0
    axiom_schemas_version:     0.1.0
    axiom_schema:              cordex-1D.json
    productive_version:        edfab29
    variable_version:          v20231001
    Conventions:               CF-1.10, ACDD-1.3
    ...                        ...
    geospatial_lat_max:        12.98
    geospatial_lat_units:      degrees_north
    geospatial_lon_min:        88.48
    geospatial_lon_max:        207.39
    geospatial_lon_units:      degrees_east
    history:                   Sat May 04 16:03:59 2024: /g/data/access/ngm/m...

In [58]:
rsds_encoding = specify_encoding(rsds_anoms)

In [59]:
rsds_encoding['rsds']

{'chunks': (17167, 319, 5),
 'compressors': BloscCodec(_tunable_attrs={'typesize'}, typesize=1, cname=<BloscCname.zstd: 'zstd'>, clevel=6, shuffle=<BloscShuffle.bitshuffle: 'bitshuffle'>, blocksize=0),
 'dtype': 'float64'}

In [60]:
write_zarr(
    rsds_anoms,
    BARRA_R2_WRITE_PATH,
    'rsds_dayofyear_anoms_1979-2025.zarr',
    rsds_encoding
)